# Bracket-Experts MoE — Chess Elo Prediction (GPU training)

8 independent GRU experts (moves + per-ply clock-time only, no engine/XGBoost
features), each expert's loss masked to its own ~400-Elo bracket, combined by
a trainable stacking layer into final white/black Elo bucket predictions.

**Before running the full thing**: run every cell with `EPOCHS = 1` first
(default below). The training-loop cell prints the real wall-clock seconds
for that epoch — use that to estimate total cost before committing to
`EPOCHS = 20`.

**Setup**: your `nn_bracket_moe` folder (`dataset.parquet` + `manifest.json`
+ `vocab.json`) needs to be somewhere in your Google Drive. Edit `DATA_DIR`
below to match wherever it landed.

**Memory**: loading reads only the ~4.7GB of arrays training actually needs
(not the full parquet's every column, and not doubled by an unnecessary
int64 cast or duplicated per train/val/test split) — this should fit in a
standard 12GB Colab runtime. If it's still tight, switch to a High-RAM
runtime: `Runtime > Change runtime type > High-RAM`.

**Checkpoints**: three kinds get saved to `OUT_DIR` every epoch —
`{tag}_last.pt` (overwritten every epoch, for resuming after a disconnect),
`{tag}_best.pt` (overwritten whenever val loss improves), and
`{tag}_epoch{NNN}.pt` (one **standalone snapshot per epoch, never
overwritten**). Download whichever epoch's file you actually want instead of
whatever `last`/`best` currently happen to point to, and resume or branch
from any specific past epoch by setting `RESUME` to that exact file.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected. Runtime > Change runtime type > GPU (T4).")

## Config — edit these

`DATA_DIR` must point at the folder containing `dataset.parquet`,
`manifest.json`, and `vocab.json`. Use the Colab file browser (folder icon,
left sidebar) under `drive/MyDrive/...` to find the exact path if unsure.

In [ ]:
from pathlib import Path

DATA_DIR = Path('/content/drive/MyDrive/nn_bracket_moe')     # EDIT to match your Drive path
OUT_DIR = Path('/content/drive/MyDrive/bracket_moe_checkpoints')
OUT_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS = 1              # start at 1 for calibration; bump to 20 for the full run
BATCH_SIZE = 512
EMBED_DIM = 24
HIDDEN_DIM = 48
LR = 1e-3
LR_MIN = 1e-5
WEIGHT_DECAY = 1e-5
WEIGHT_POWER = 0.5
EXPERT_LOSS_WEIGHT = 1.0
MODEL_TAG = "bracket_moe_gpu"
RESUME = None            # set to a string path, e.g. str(OUT_DIR / f"{MODEL_TAG}_last.pt"), to resume
CHECKPOINT_EVERY = 1     # also save a standalone, never-overwritten snapshot every N epochs
                         # ({MODEL_TAG}_epoch{NNN}.pt) - last/best get overwritten every epoch they
                         # change, so if you download one mid-run and training later moves past it,
                         # there's no way to tell which epoch you actually have. Download whichever
                         # epoch file you actually want, or resume/branch from any specific past
                         # epoch with RESUME = str(OUT_DIR / f"{MODEL_TAG}_epoch007.pt")

assert (DATA_DIR / "dataset.parquet").exists(), f"dataset.parquet not found in {DATA_DIR} - fix DATA_DIR above"
assert (DATA_DIR / "manifest.json").exists(), f"manifest.json not found in {DATA_DIR} - fix DATA_DIR above"
print("Data dir OK:", DATA_DIR)

## Model definition (identical to `scripts/train_bracket_moe_gpu.py`)

In [ ]:
import gc
import json as json_lib
import time
import numpy as np
import pyarrow.parquet as pq
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset

NUM_BUCKETS = 14
BUCKET_LO, BUCKET_WIDTH = 400, 200
BUCKET_MIDPOINTS = np.array([BUCKET_LO + BUCKET_WIDTH * i + BUCKET_WIDTH / 2 for i in range(NUM_BUCKETS)])
NUM_EXPERTS = 8
RANDOM_STATE = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

In [ ]:
class BracketDataset(Dataset):
    def __init__(self, tokens, lengths, time_spent, white_bucket, black_bucket, white_masks, black_masks):
        # tokens stays int32 here (half the size of int64) - nn.Embedding only
        # needs int64 indices for the current batch, cast lazily in Expert.forward,
        # not for the whole multi-million-row array up front (that doubling was
        # the single biggest cause of the RAM blowup on Colab).
        self.tokens = torch.from_numpy(tokens)
        self.lengths = torch.from_numpy(lengths).long().clamp(min=1)
        self.time_spent = torch.from_numpy(time_spent).float()
        self.white_bucket = torch.from_numpy(white_bucket).long()
        self.black_bucket = torch.from_numpy(black_bucket).long()
        self.white_masks = torch.from_numpy(white_masks).float()
        self.black_masks = torch.from_numpy(black_masks).float()

    def __len__(self):
        return len(self.white_bucket)

    def __getitem__(self, idx):
        return (self.tokens[idx], self.lengths[idx], self.time_spent[idx],
                self.white_bucket[idx], self.black_bucket[idx],
                self.white_masks[idx], self.black_masks[idx])


class Expert(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_buckets=NUM_BUCKETS, mlp_hidden=48, dropout=0.2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim + 1, hidden_dim, batch_first=True)
        self.white_head = nn.Sequential(nn.Linear(hidden_dim, mlp_hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(mlp_hidden, num_buckets))
        self.black_head = nn.Sequential(nn.Linear(hidden_dim, mlp_hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(mlp_hidden, num_buckets))

    def forward(self, tokens, lengths, time_spent):
        emb = self.embed(tokens.long())
        x = torch.cat([emb, time_spent.unsqueeze(-1)], dim=-1)
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h_n = self.gru(packed)
        h = h_n[-1]
        return self.white_head(h), self.black_head(h)


class BracketExpertsMoE(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_experts=NUM_EXPERTS, num_buckets=NUM_BUCKETS,
                 expert_mlp_hidden=48, combiner_hidden=96, dropout=0.2):
        super().__init__()
        self.experts = nn.ModuleList([
            Expert(vocab_size, embed_dim, hidden_dim, num_buckets, expert_mlp_hidden, dropout)
            for _ in range(num_experts)
        ])
        self.num_experts = num_experts
        self.num_buckets = num_buckets
        combiner_in = num_experts * (num_buckets + 1)
        self.register_buffer("midpoints", torch.from_numpy(BUCKET_MIDPOINTS).float())
        self.white_combiner = nn.Sequential(nn.Linear(combiner_in, combiner_hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(combiner_hidden, num_buckets))
        self.black_combiner = nn.Sequential(nn.Linear(combiner_in, combiner_hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(combiner_hidden, num_buckets))

    def forward(self, tokens, lengths, time_spent):
        white_logits_list, black_logits_list = [], []
        white_probs_list, black_probs_list = [], []
        for expert in self.experts:
            wl, bl = expert(tokens, lengths, time_spent)
            white_logits_list.append(wl)
            black_logits_list.append(bl)
            white_probs_list.append(torch.softmax(wl, dim=-1))
            black_probs_list.append(torch.softmax(bl, dim=-1))

        white_probs = torch.stack(white_probs_list, dim=1)
        black_probs = torch.stack(black_probs_list, dim=1)
        white_guess = (white_probs * self.midpoints).sum(dim=-1)
        black_guess = (black_probs * self.midpoints).sum(dim=-1)

        white_combiner_in = torch.cat([white_probs.flatten(start_dim=1), white_guess], dim=1)
        black_combiner_in = torch.cat([black_probs.flatten(start_dim=1), black_guess], dim=1)
        final_white = self.white_combiner(white_combiner_in)
        final_black = self.black_combiner(black_combiner_in)

        return final_white, final_black, white_logits_list, black_logits_list

In [ ]:
def run_epoch(model, loader, optimizer, criterion, device, train, expert_loss_weight=1.0, grad_clip=1.0):
    model.train(train)
    total_loss, n = 0.0, 0
    correct_w, correct_b, adj_w, adj_b = 0, 0, 0, 0
    ce_none = nn.CrossEntropyLoss(reduction="none")
    for tok, length, tspent, white_y, black_y, white_masks, black_masks in loader:
        tok, length, tspent = tok.to(device), length.to(device), tspent.to(device)
        white_y, black_y = white_y.to(device), black_y.to(device)
        white_masks, black_masks = white_masks.to(device), black_masks.to(device)

        if train:
            optimizer.zero_grad()
        final_white, final_black, white_logits_list, black_logits_list = model(tok, length, tspent)

        combiner_loss = criterion(final_white, white_y) + criterion(final_black, black_y)

        expert_loss = 0.0
        for i in range(model.num_experts):
            w_ce = ce_none(white_logits_list[i], white_y)
            b_ce = ce_none(black_logits_list[i], black_y)
            w_mask = white_masks[:, i]
            b_mask = black_masks[:, i]
            w_denom = w_mask.sum().clamp(min=1)
            b_denom = b_mask.sum().clamp(min=1)
            expert_loss = expert_loss + (w_ce * w_mask).sum() / w_denom + (b_ce * b_mask).sum() / b_denom
        expert_loss = expert_loss / model.num_experts

        loss = combiner_loss + expert_loss_weight * expert_loss
        if train:
            loss.backward()
            if grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

        bs = len(white_y)
        total_loss += loss.item() * bs
        n += bs
        with torch.no_grad():
            pred_w, pred_b = final_white.argmax(1), final_black.argmax(1)
            correct_w += (pred_w == white_y).sum().item()
            correct_b += (pred_b == black_y).sum().item()
            adj_w += ((pred_w - white_y).abs() <= 1).sum().item()
            adj_b += ((pred_b - black_y).abs() <= 1).sum().item()
    return {"loss": total_loss / n, "acc_w": correct_w / n, "acc_b": correct_b / n,
            "adj_w": adj_w / n, "adj_b": adj_b / n}


def fmt(m):
    return f"loss={m['loss']:.3f} acc(w/b)={m['acc_w']:.3f}/{m['acc_b']:.3f} adj_acc(w/b)={m['adj_w']:.3f}/{m['adj_b']:.3f}"


def _fixed_list_to_2d(record_batch, col_name, width, dtype):
    flat = record_batch.column(col_name).combine_chunks().flatten().to_numpy(zero_copy_only=False)
    return flat.reshape(-1, width).astype(dtype, copy=False)


def load_parquet_dataset(data_dir, max_len, num_experts):
    # Streams the file row-group by row-group (~50k rows each, matching how
    # preprocess_bracket_moe_data.py wrote it) into pre-allocated output
    # arrays, instead of pq.read_table()-ing the whole ~4M-row file at once.
    # A single read_table() + combine_chunks()/astype() on the full file
    # briefly holds 3-4 full-size copies of the largest columns in memory
    # simultaneously (source table, combined-chunk copy, astype() copy -
    # astype() always copies unless told not to) - comfortably blows past a
    # 12GB Colab runtime even though the final arrays only total ~4.7GB.
    needed_cols = ["tokens", "time_spent", "white_masks", "black_masks", "length", "white_bucket", "black_bucket"]
    pf = pq.ParquetFile(data_dir / "dataset.parquet")
    n = pf.metadata.num_rows

    tokens = np.empty((n, max_len), dtype=np.int32)
    time_spent = np.empty((n, max_len), dtype=np.float32)
    white_masks = np.empty((n, num_experts), dtype=np.float32)
    black_masks = np.empty((n, num_experts), dtype=np.float32)
    lengths = np.empty(n, dtype=np.int32)
    white_bucket = np.empty(n, dtype=np.int64)
    black_bucket = np.empty(n, dtype=np.int64)

    offset = 0
    for rg_idx in range(pf.num_row_groups):
        rg = pf.read_row_group(rg_idx, columns=needed_cols)
        m = rg.num_rows
        sl = slice(offset, offset + m)
        tokens[sl] = _fixed_list_to_2d(rg, "tokens", max_len, np.int32)
        time_spent[sl] = _fixed_list_to_2d(rg, "time_spent", max_len, np.float32)
        white_masks[sl] = _fixed_list_to_2d(rg, "white_masks", num_experts, np.float32)
        black_masks[sl] = _fixed_list_to_2d(rg, "black_masks", num_experts, np.float32)
        lengths[sl] = rg.column("length").to_numpy().astype(np.int32, copy=False)
        white_bucket[sl] = rg.column("white_bucket").to_numpy().astype(np.int64, copy=False)
        black_bucket[sl] = rg.column("black_bucket").to_numpy().astype(np.int64, copy=False)
        offset += m
        del rg
    return tokens, lengths, time_spent, white_bucket, black_bucket, white_masks, black_masks

## Load data

In [ ]:
torch.manual_seed(RANDOM_STATE)

manifest = json_lib.loads((DATA_DIR / "manifest.json").read_text())
vocab_size = manifest["vocab_size"]

print(f"Loading pre-processed dataset from {DATA_DIR}...")
t0 = time.time()
tokens, lengths, time_spent, white_bucket, black_bucket, white_masks, black_masks = load_parquet_dataset(
    DATA_DIR, manifest["max_len"], manifest["num_experts"],
)
print(f"  {len(tokens):,} games, vocab_size={vocab_size:,}, loaded in {time.time()-t0:.1f}s")
print(f"  white bracket coverage: {manifest['white_bracket_coverage']}")
print(f"  black bracket coverage: {manifest['black_bracket_coverage']}")
gc.collect()  # release the intermediate pyarrow Table's buffers promptly

n = len(white_bucket)
rng = np.random.RandomState(RANDOM_STATE)
perm = rng.permutation(n)
n_test, n_val = int(n * 0.1), int(n * 0.1)
test_idx, val_idx, train_idx = perm[:n_test], perm[n_test:n_test + n_val], perm[n_test + n_val:]
print(f"train={len(train_idx):,} val={len(val_idx):,} test={len(test_idx):,}")

combined_targets = np.concatenate([white_bucket[train_idx], black_bucket[train_idx]])
counts = np.clip(np.bincount(combined_targets, minlength=NUM_BUCKETS).astype(np.float64), 1, None)
weights = (counts.sum() / (NUM_BUCKETS * counts)) ** WEIGHT_POWER
class_weights = torch.tensor(weights, dtype=torch.float32, device=device)

# One dataset over the full arrays, sliced via Subset (index-only, no copy)
# instead of building train/val/test as three separately fancy-indexed numpy
# copies - fancy indexing (tokens[idx]) allocates a brand new array, so three
# eagerly-copied splits plus the original full array would coexist in memory
# simultaneously, nearly doubling the largest arrays' footprint for no reason.
full_dataset = BracketDataset(tokens, lengths, time_spent, white_bucket, black_bucket, white_masks, black_masks)
pin = device.type == "cuda"
train_loader = DataLoader(Subset(full_dataset, train_idx), batch_size=BATCH_SIZE, shuffle=True, pin_memory=pin, num_workers=2)
val_loader = DataLoader(Subset(full_dataset, val_idx), batch_size=1024, pin_memory=pin, num_workers=2)
test_loader = DataLoader(Subset(full_dataset, test_idx), batch_size=1024, pin_memory=pin, num_workers=2)

## Build model

In [ ]:
model = BracketExpertsMoE(vocab_size=vocab_size, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)
criterion = nn.CrossEntropyLoss(weight=class_weights)

start_epoch = 1
best_val_loss = float("inf")
if RESUME:
    ckpt = torch.load(RESUME, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    scheduler.T_max = EPOCHS  # keep THIS run's EPOCHS as the cosine horizon, not
    # whatever the checkpoint happened to be trained under (e.g. a short calibration
    # run) - a mismatched T_max would otherwise silently wreck the LR schedule
    start_epoch = ckpt["epoch"] + 1
    best_val_loss = ckpt["best_val_loss"]
    print(f"Resumed from {RESUME} at epoch {start_epoch}")

ckpt_path = OUT_DIR / f"{MODEL_TAG}_best.pt"
last_path = OUT_DIR / f"{MODEL_TAG}_last.pt"
print(f"Ready to train {NUM_EXPERTS}-expert bracket MoE for {EPOCHS} epochs (starting at epoch {start_epoch}).")

## Train

Checkpoints save to Drive every epoch (`*_last.pt` always, `*_best.pt` on
val-loss improvement), so a disconnect mid-run doesn't lose progress —
re-run this notebook with `RESUME` set to `*_last.pt`'s path and it picks up
where it left off.

**Watch the printed `(NNN.Ns)` per epoch** — that's your real per-epoch
wall-clock time for cost estimation.

In [ ]:
for epoch in range(start_epoch, EPOCHS + 1):
    t0 = time.time()
    train_m = run_epoch(model, train_loader, optimizer, criterion, device, train=True, expert_loss_weight=EXPERT_LOSS_WEIGHT)
    with torch.no_grad():
        val_m = run_epoch(model, val_loader, optimizer, criterion, device, train=False, expert_loss_weight=EXPERT_LOSS_WEIGHT)
    scheduler.step()
    improved = val_m["loss"] < best_val_loss
    print(f"  epoch {epoch}: train[{fmt(train_m)}] val[{fmt(val_m)}] ({time.time()-t0:.1f}s){' *' if improved else ''}", flush=True)

    if improved:
        best_val_loss = val_m["loss"]
    # build ckpt_dict AFTER best_val_loss is finalized for this epoch, so
    # last_path/ckpt_path/the per-epoch snapshot all agree on it
    ckpt_dict = {
        "model": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
        "epoch": epoch, "best_val_loss": best_val_loss,
    }
    torch.save(ckpt_dict, last_path)
    if improved:
        torch.save(ckpt_dict, ckpt_path)
    # standalone per-epoch snapshot, never overwritten - see CHECKPOINT_EVERY note above
    if CHECKPOINT_EVERY > 0 and epoch % CHECKPOINT_EVERY == 0:
        torch.save(ckpt_dict, OUT_DIR / f"{MODEL_TAG}_epoch{epoch:03d}.pt")

print("Done training.")

## Final test evaluation (best checkpoint)

In [ ]:
best = torch.load(ckpt_path, map_location=device)
model.load_state_dict(best["model"])
with torch.no_grad():
    test_m = run_epoch(model, test_loader, optimizer, criterion, device, train=False, expert_loss_weight=EXPERT_LOSS_WEIGHT)
print(f"Final test (best val checkpoint, epoch {best['epoch']}): {fmt(test_m)}")

metrics_path = OUT_DIR / f"{MODEL_TAG}_metrics.json"
metrics_path.write_text(json_lib.dumps({
    "model": f"BracketExpertsMoE_{MODEL_TAG}", "n_train": len(train_idx), "n_val": len(val_idx), "n_test": len(test_idx),
    "vocab_size": vocab_size, "epochs": EPOCHS, "best_epoch": best["epoch"], "test_metrics": test_m,
    "embed_dim": EMBED_DIM, "hidden_dim": HIDDEN_DIM, "num_experts": NUM_EXPERTS,
}, indent=2))
print(f"Saved: {metrics_path}")
print(f"Checkpoint: {ckpt_path}")